In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
%sql
INSERT INTO saleslake_${environment}.silver_schema_${environment}.cleanedSales
SELECT
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    ingest_ts
FROM (
    SELECT
        CAST(TRIM(sale_id) AS INTEGER) AS sale_id,
        UPPER(TRIM(product)) AS product,
        UPPER(TRIM(category)) AS category,
        CAST(TRIM(quantity) AS INTEGER) AS quantity,
        CAST(TRIM(price) AS DOUBLE) AS price,
        TO_DATE(TRIM(sale_date), 'yyyy-MM-dd') AS sale_date,
        UPPER(TRIM(region)) AS region,
        current_timestamp AS ingest_ts,
        
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(sale_id)
            ORDER BY ingest_ts DESC
        ) AS rn

    FROM saleslake_${environment}.bronze_schema_${environment}.rawSales
    
    WHERE ingest_ts > (
        SELECT COALESCE(
            MAX(ingest_ts),
            TO_TIMESTAMP('1900-01-01', 'yyyy-MM-dd')
        )
        FROM saleslake_${environment}.silver_schema_${environment}.cleanedSales
    )
) t
WHERE rn = 1
ORDER BY sale_id;